In [ ]:

def recommendation_agent(risk, income):

    if risk == "Aggressive":

        return {
            "sip": income * 0.1,
            "fd": 0,
            "insurance": "Term Insurance"
        }

    elif risk == "Moderate":

        return {
            "sip": income * 0.05,
            "fd": income * 0.1,
            "insurance": "Health Insurance"
        }

    else:

        return {
            "sip": income * 0.03,
            "fd": income * 0.2,
            "insurance": "Life Insurance"
        }

In [15]:
df = pd.read_csv(
    r"C:\Users\yashg\Downloads\MODELS\sbi-hackathon\sbi-hackathon\data\customer_profile\customer_profiles.csv"
)
df.head(5)

,customer_id,name,phone_number,age,profession,income,marital_status,location,total_expenses,investment_amount,...,Food,Healthcare,Investment,Shopping,Travel,Utilities,savings,savings_rate,goals,risk
0,C0001,Megha Parikh,9851352150,62,Teacher,132838,Married,Agra,185298,24644.0,...,63335,19821,24644,6674,17029,12509,-77104.0,-58.04,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative
1,C0002,Ria Natarajan,9907425858,63,Teacher,54427,Single,Hyderabad,69855,10331.0,...,6975,15354,10331,2066,12436,14989,-25759.0,-47.33,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative
2,C0003,Ayaan Dhar,9272838699,48,Government Employee,201296,Single,Mumbai,185720,24890.0,...,21979,21396,24890,39693,4167,50169,-9314.0,-4.63,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Moderate
3,C0004,Pallavi Sarna,7481517297,65,Chartered Accountant,177750,Married,Delhi,134228,29675.0,...,31494,41652,29675,11650,13121,6177,13847.0,7.79,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative
4,C0005,Christopher Rao,8583365498,61,Teacher,56088,Single,Mumbai,79835,0.0,...,0,20815,0,11046,13769,15467,-23747.0,-42.34,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative


In [1]:
import pandas as pd

In [3]:
import os

print(os.path.exists("data/customer_profile/customer_profiles.csv"))

False


In [7]:
risk = df['risk'].iloc[0]
goals = df['goals'].iloc[0]
income = df['income'].iloc[0]
savings = df['savings_rate'].iloc[0]

In [25]:
def recommendation_agent(profile):

    investable_amount = profile["savings"] * 0.8

    recommendations = []

    for goal in profile['goals']:

        if goal == "Build Emergency Fund":
            product = "Fixed Deposit"
            allocation = 0.25

        elif goal == "Long-Term Wealth Creation":
            if risk == "Aggressive":
                product = "Equity Mutual Fund SIP"
            elif risk == "Moderate":
                product = "Balanced Fund SIP"
            else:
                product = "Debt Fund"
            allocation = 0.35

        elif goal == "Retirement Planning":
            if risk == "Aggressive":
                product = "Equity Mutual Fund SIP"
            elif risk == "Moderate":
                product = "Balanced Fund SIP"
            else:
                product = "PPF"
            allocation = 0.30

        elif goal == "Home Purchase":
            product = "Recurring Deposit"
            allocation = 0.25

        elif goal == "Child Education Planning":
            product = "Hybrid Mutual Fund"
            allocation = 0.25

        elif goal == "Vehicle Purchase":
            product = "Recurring Deposit"
            allocation = 0.20

        else:
            product = "Savings Account"
            allocation = 0.10

    recommendations.append({
        "goal": goal,
        "product": product,
        "monthly_amount": round(investable_amount * allocation)
    })

    return recommendations

In [26]:
recommendation_agent(df.iloc[0])

[{'goal': ']', 'product': 'Savings Account', 'monthly_amount': -6168}]

In [24]:
recommendation_agent(df.iloc[9])

[{'goal': ']', 'product': 'Savings Account', 'monthly_amount': -1181}]

In [27]:
df['goals']

0      [{"goal": "Build Emergency Fund", "confidence"...
1      [{"goal": "Build Emergency Fund", "confidence"...
2      [{"goal": "Build Emergency Fund", "confidence"...
3      [{"goal": "Build Emergency Fund", "confidence"...
4      [{"goal": "Build Emergency Fund", "confidence"...
                             ...                        
995    [{"goal": "Long-Term Wealth Creation", "confid...
996    [{"goal": "Build Emergency Fund", "confidence"...
997    [{"goal": "Build Emergency Fund", "confidence"...
998    [{"goal": "Build Emergency Fund", "confidence"...
999    [{"goal": "Build Emergency Fund", "confidence"...
Name: goals, Length: 1000, dtype: str

In [43]:
import json

def recommendation_agent(row):

    income = row["income"]
    total_expenses = row["total_expenses"]
    risk = row["risk"]

    # Calculate savings
    monthly_savings = max(0, row['savings'])
    
    savings_rate = row['savings_rate']

    # Invest only 80% of savings
    investable_amount = monthly_savings * 0.8

    # Parse goals JSON
    goal_str = row["goals"]

    if isinstance(goal_str, str):
        goal_str = goal_str.replace('""', '"')
        goals = json.loads(goal_str)
    else:
        goals = goal_str

    recommendations = []

    total_confidence = sum(
        goal["confidence"] for goal in goals
    )

    for goal_data in goals:

        goal = goal_data["goal"]
        confidence = goal_data["confidence"]

        amount = round(
            investable_amount *
            (confidence / total_confidence)
        )

        # Product recommendation
        if goal == "Build Emergency Fund":

            product = "Fixed Deposit / Liquid Fund"

        elif goal == "Long-Term Wealth Creation":

            if risk == "Aggressive":
                product = "Equity Mutual Fund SIP"
            elif risk == "Moderate":
                product = "Balanced Mutual Fund SIP"
            else:
                product = "Debt Mutual Fund"

        elif goal == "Retirement Planning":

            if risk == "Aggressive":
                product = "Equity Mutual Fund SIP"
            elif risk == "Moderate":
                product = "Balanced Mutual Fund SIP"
            else:
                product = "PPF"

        elif goal == "Home Purchase":

            product = "Recurring Deposit"

        elif goal == "Child Education Planning":

            product = "Hybrid Mutual Fund"

        elif goal == "Vehicle Purchase":

            product = "Recurring Deposit"

        else:

            product = "Savings Account"

        recommendations.append({
            "goal": goal,
            "confidence": confidence,
            "product": product,
            "monthly_amount": amount
        })

    # Voice summary
    summary = (
        f"Based on your monthly income of ₹{income:,.0f}, "
        f"savings rate of {savings_rate:.1f}% "
        f"and {risk.lower()} risk profile, "
        f"we recommend the following investment strategy. "
    )

    for rec in recommendations:

        summary += (
            f"For {rec['goal']}, invest approximately "
            f"₹{rec['monthly_amount']:,.0f} per month in "
            f"{rec['product']}. "
        )

    return {
        "recommendations": recommendations
    }

In [45]:
df["recommendation"] = df.apply(
    recommendation_agent,
    axis=1
)

In [46]:
df

,customer_id,name,phone_number,age,profession,income,marital_status,location,total_expenses,investment_amount,...,Healthcare,Investment,Shopping,Travel,Utilities,savings,savings_rate,goals,risk,recommendation
0,C0001,Megha Parikh,9851352150,62,Teacher,132838,Married,Agra,185298,24644.0,...,19821,24644,6674,17029,12509,-77104.0,-58.04,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative,{'recommendations': [{'goal': 'Build Emergency...
1,C0002,Ria Natarajan,9907425858,63,Teacher,54427,Single,Hyderabad,69855,10331.0,...,15354,10331,2066,12436,14989,-25759.0,-47.33,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative,{'recommendations': [{'goal': 'Build Emergency...
2,C0003,Ayaan Dhar,9272838699,48,Government Employee,201296,Single,Mumbai,185720,24890.0,...,21396,24890,39693,4167,50169,-9314.0,-4.63,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Moderate,{'recommendations': [{'goal': 'Build Emergency...
3,C0004,Pallavi Sarna,7481517297,65,Chartered Accountant,177750,Married,Delhi,134228,29675.0,...,41652,29675,11650,13121,6177,13847.0,7.79,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative,{'recommendations': [{'goal': 'Build Emergency...
4,C0005,Christopher Rao,8583365498,61,Teacher,56088,Single,Mumbai,79835,0.0,...,20815,0,11046,13769,15467,-23747.0,-42.34,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative,{'recommendations': [{'goal': 'Build Emergency...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,C0996,Oscar Murty,9959114849,46,Farmer,139648,Married,Mumbai,57418,21804.0,...,9879,21804,12617,16170,13082,60426.0,43.27,"[{""goal"": ""Long-Term Wealth Creation"", ""confid...",Moderate,{'recommendations': [{'goal': 'Long-Term Wealt...
996,C0997,Sathvik Shanker,6988169835,58,Teacher,164387,Single,Chennai,146023,17028.0,...,22565,17028,45006,17660,0,1336.0,0.81,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Moderate,{'recommendations': [{'goal': 'Build Emergency...
997,C0998,Ishani Sekhon,6807987800,64,Business Owner,59513,Single,Varanasi,189418,697.0,...,12282,697,46149,22408,19356,-130602.0,-219.45,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative,{'recommendations': [{'goal': 'Build Emergency...
998,C0999,Sachi Mannan,8191853649,58,Student,42073,Married,Chennai,181257,20113.0,...,17613,20113,24428,23539,20885,-159297.0,-378.62,"[{""goal"": ""Build Emergency Fund"", ""confidence""...",Conservative,{'recommendations': [{'goal': 'Build Emergency...
